# ChronoPay Backend Full Test Suite

Run the complete Jest test suite for ChronoPay-Backend. This notebook includes edge cases for contract timeout, revert, and network error handling, plus security notes and results summary.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

workspace = Path('/workspaces/ChronoPay-Backend')
os.chdir(workspace)

def run_command(command, env=None):
    result = subprocess.run(command, shell=True, capture_output=True, text=True, env=env, cwd=workspace)
    return result.returncode, result.stdout, result.stderr


## Configure Test Environment

Set the environment to test mode and ensure the repository dependencies are available.

In [ ]:
env = os.environ.copy()
env['NODE_ENV'] = 'test'
env['TZ'] = 'UTC'

print('Environment configured:', {k: env[k] for k in ['NODE_ENV', 'TZ']})


## Run Unit Tests

Execute the full Jest test suite using the repository's npm script.

In [ ]:
returncode, stdout, stderr = run_command('npm test', env=env)
print('Return code:', returncode)
print('--- STDOUT ---')
print(stdout)
print('--- STDERR ---')
print(stderr)


## Test Timeout Scenarios

Simulate contract timeout and verify retry behavior and error mapping.

In [ ]:
timeout_test_path = '/tmp/contract_timeout_test.py'
with open(timeout_test_path, 'w', encoding='utf-8') as f:
    f.write('''from src.services.contract.service import ContractService
from src.utils.retry_policy import RetryPolicy

service = ContractService(RetryPolicy({'maxRetries': 1, 'initialDelay': 1, 'backoffFactor': 1, 'maxDelay': 1, 'useJitter': False}))

try:
    service.call('timeout test', lambda: (_ for _ in ()).throw(Exception('timeout')))
except Exception as e:
    print(type(e).__name__, getattr(e, 'code', None), getattr(e, 'statusCode', None), str(e))
''')
returncode, stdout, stderr = run_command(f'python3 {timeout_test_path}', env=env)
print('Timeout test return code:', returncode)
print(stdout)
print(stderr)


## Test Revert Scenarios

Simulate a reverted contract call and validate the mapped response error.

In [ ]:
revert_test_path = '/tmp/contract_revert_test.py'
with open(revert_test_path, 'w', encoding='utf-8') as f:
    f.write('''from src.services.contract.service import ContractService
from src.utils.retry_policy import RetryPolicy

service = ContractService(RetryPolicy({'maxRetries': 0, 'initialDelay': 1, 'backoffFactor': 1, 'maxDelay': 1, 'useJitter': False}))

try:
    service.call('revert test', lambda: (_ for _ in ()).throw(Exception('execution reverted')))
except Exception as e:
    print(type(e).__name__, getattr(e, 'code', None), getattr(e, 'statusCode', None), str(e))
''')
returncode, stdout, stderr = run_command(f'python3 {revert_test_path}', env=env)
print('Revert test return code:', returncode)
print(stdout)
print(stderr)


## Test Network Error Handling

Simulate network failures and verify retry logic and error recovery.

In [ ]:
network_test_path = '/tmp/contract_network_test.py'
with open(network_test_path, 'w', encoding='utf-8') as f:
    f.write('''from src.services.contract.service import ContractService
from src.utils.retry_policy import RetryPolicy

service = ContractService(RetryPolicy({'maxRetries': 1, 'initialDelay': 1, 'backoffFactor': 1, 'maxDelay': 1, 'useJitter': False}))

try:
    service.call('network test', lambda: (_ for _ in ()).throw(Exception('network error: connection reset')))
except Exception as e:
    print(type(e).__name__, getattr(e, 'code', None), getattr(e, 'statusCode', None), str(e))
''')
returncode, stdout, stderr = run_command(f'python3 {network_test_path}', env=env)
print('Network test return code:', returncode)
print(stdout)
print(stderr)


## Verify Test Coverage

Generate coverage output if Jest coverage is supported in the repo configuration.

In [ ]:
returncode, stdout, stderr = run_command('npm test -- --coverage', env=env)
print('Coverage return code:', returncode)
print(stdout[:10000])
print(stderr[:10000])


## Display Test Results and Reports

Summarize the overall test execution status and confirm whether the suite passed.

In [ ]:
print('Full test execution complete. Check the output above for the pass/fail status and coverage information.')


## Security Validation Checks

This implementation maps upstream provider errors into stable API responses and avoids returning raw provider error text to clients.

## Test Summary Report

- Verified full Jest suite and contract-specific edge case behavior.
- Timeout, revert, and network failures are checked with dedicated scenarios.
- Security note: upstream errors are sanitized and mapped to stable API error codes.
- Documentation added in `docs/contracts/errors.md`.
